# Xarray-Spatial Performance: Fused Overlap

Chaining spatial operations like erode-then-dilate adds a separate blockwise layer to the Dask graph for each step. `fused_overlap` collapses the chain into a single `map_overlap` call, cutting scheduler overhead and memory traffic. `multi_overlap` does the same trick when one kernel produces multiple output bands.

### What you'll build

1. [Create a chunked random raster](#Data)
2. [Chain two kernels with `fused_overlap` and compare task counts](#fused_overlap:-chained-operations-in-one-pass)
3. [Produce multiple outputs from one kernel with `multi_overlap`](#multi_overlap:-N-outputs-in-one-pass)
4. [Use the `.xrs` accessor shorthand](#Accessor-syntax)

Standard imports plus the two utility functions.

In [ ]:
import numpy as np
import dask.array as da
import xarray as xr
import matplotlib.pyplot as plt
import xrspatial
from xrspatial.utils import fused_overlap, multi_overlap

## Data

A 512x512 random raster backed by Dask, chunked into 128x128 tiles.

In [ ]:
np.random.seed(42)
raw = np.random.rand(512, 512).astype(np.float32)
dem = xr.DataArray(da.from_array(raw, chunks=128), dims=['y', 'x'])
dem

16 chunks (4x4 grid) of 128x128 pixels each.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
dem.plot.imshow(ax=ax, cmap='viridis')
ax.set_title('Random input raster')
plt.tight_layout()
plt.show()

## fused_overlap: chained operations in one pass

Define two stage functions. Each takes a padded chunk and returns the unpadded interior.

In [ ]:
def smooth_interior(chunk):
    """3x3 mean filter. Takes (H+2, W+2), returns (H, W)."""
    from numpy.lib.stride_tricks import sliding_window_view
    windows = sliding_window_view(chunk, (3, 3))
    return np.nanmean(windows, axis=(-2, -1))

def threshold_interior(chunk):
    """Binary threshold. Takes (H+2, W+2), returns (H, W)."""
    interior = chunk[1:-1, 1:-1]
    return (interior > 0.5).astype(np.float32)

In [ ]:
# Fused: one map_overlap call
fused = fused_overlap(dem, (smooth_interior, 1), (threshold_interior, 1))

# Sequential: two map_overlap calls
step1 = dem.data.map_overlap(smooth_interior, depth=1, boundary=np.nan, trim=False, meta=np.array(()))
sequential = step1.map_overlap(threshold_interior, depth=1, boundary=np.nan, trim=False, meta=np.array(()))

print(f'Fused graph:      {len(dict(fused.data.__dask_graph__())):,} tasks')
print(f'Sequential graph: {len(dict(sequential.__dask_graph__())):,} tasks')

The fused version builds fewer graph tasks because both kernels run inside a single `map_overlap` call rather than stacking two separate overlap-and-trim cycles.

## multi_overlap: N outputs in one pass

When a kernel returns a 3D array `(N, H, W)`, `multi_overlap` splits the bands into separate DataArray slices along a new leading dimension.

In [ ]:
def gradient_kernel(chunk):
    """Compute dx and dy gradients. Takes (H+2, W+2), returns (2, H, W)."""
    dx = (chunk[1:-1, 2:] - chunk[1:-1, :-2]) / 2.0
    dy = (chunk[2:, 1:-1] - chunk[:-2, 1:-1]) / 2.0
    return np.stack([dx, dy], axis=0)

result = multi_overlap(dem, gradient_kernel, n_outputs=2, depth=1)
print(f'Output shape: {result.shape}')
print(f'Dimensions:   {result.dims}')
print(f'Graph tasks:  {len(dict(result.data.__dask_graph__())):,}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
result[0].plot.imshow(ax=axes[0], cmap='RdBu', robust=True)
axes[0].set_title('dx gradient')
result[1].plot.imshow(ax=axes[1], cmap='RdBu', robust=True)
axes[1].set_title('dy gradient')
plt.tight_layout()
plt.show()

## Accessor syntax

Both functions are available through the `.xrs` accessor on any DataArray.

In [ ]:
fused_acc = dem.xrs.fused_overlap((smooth_interior, 1), (threshold_interior, 1))
multi_acc = dem.xrs.multi_overlap(gradient_kernel, n_outputs=2, depth=1)
print('Accessor: OK')

### References

- [Dask `map_overlap` documentation](https://docs.dask.org/en/stable/generated/dask.array.overlap.map_overlap.html)
- [xrspatial API reference](https://xarray-spatial.readthedocs.io/en/latest/reference.html)
- [Dask best practices: graph optimization](https://docs.dask.org/en/stable/best-practices.html)